In [1]:
import pandas as pd
import numpy as np
import requests

header = {"Authorization": "31WmFjhG0zzzyqT7LyXfuTS9sHopnzU90kxnLRFSPAuwE8w4N7kg6zuPeBRs"}

def make_square_geometry(lat, lon, size_km=1):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset
    ]

sites = {
    "San Simon Valley":        (32.17119355, -109.1578372),
    "Butler Valley":           (33.95808089, -113.7841023),
    "McMulllen Valley":        (33.95725, -113.0803889),
    "Hualapai Valley Basin":   (35.47122222, -113.9758333),
    "Southern Willcox Basin":  (31.36597222,-109.6629444),
    "Nevada (Amargosa)":       (36.55634, -116.496),
    "Arizona (Willcox Basin)": (32.03619, -109.754),
    "Cuyama":                  (34.89025548, -119.6165171)
}

dfs = {}

for name, (lat, lon) in sites.items():
    print(f"Querying {name}...")

    polygon_args = {
        "date_range": ["2010-01-01", "2019-12-31"],
        "interval": "monthly",
        "geometry": make_square_geometry(lat, lon, size_km=1),
        "model": "SSEBop",
        "reducer": "mean",
        "variable": "ET",
        "reference_et": "gridMET",
        "units": "mm",
        "file_format": "JSON",
        "version": 2.0
    }

    resp = requests.post(
        headers=header,
        json=polygon_args,
        url="https://openet-api.org/raster/timeseries/polygon"
    )

    print(f"  Status: {resp.status_code}")

    data = resp.json()

    if resp.status_code != 200:
        print(f"  ERROR for {name}: {data}")
        continue  # skip to next site instead of crashing

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        print(f"  Unexpected dict response for {name}: {data}")
        df = pd.DataFrame([data])

    df["site"] = name  # tag each row with the site name
    dfs[name] = df
    print(f"  Got {len(df)} rows")

# Combine all sites into one DataFrame
df_monthly = pd.concat(dfs.values(), ignore_index=True)
print(df_monthly.head())

Querying San Simon Valley...
  Status: 200
  Got 120 rows
Querying Butler Valley...
  Status: 200
  Got 120 rows
Querying McMulllen Valley...
  Status: 200
  Got 120 rows
Querying Hualapai Valley Basin...
  Status: 200
  Got 120 rows
Querying Southern Willcox Basin...
  Status: 200
  Got 120 rows
Querying Nevada (Amargosa)...
  Status: 200
  Got 120 rows
Querying Arizona (Willcox Basin)...
  Status: 200
  Got 120 rows
Querying Cuyama...
  Status: 200
  Got 120 rows
         time      et              site
0  2010-01-01  10.816  San Simon Valley
1  2010-02-01  12.648  San Simon Valley
2  2010-03-01  40.980  San Simon Valley
3  2010-04-01  60.125  San Simon Valley
4  2010-05-01  58.154  San Simon Valley


In [2]:
# Convert time column to datetime, extract year
df_monthly["time"] = pd.to_datetime(df_monthly["time"])
df_monthly["year"] = df_monthly["time"].dt.year
df_monthly

,time,et,site,year
0,2010-01-01,10.816,San Simon Valley,2010
1,2010-02-01,12.648,San Simon Valley,2010
2,2010-03-01,40.980,San Simon Valley,2010
3,2010-04-01,60.125,San Simon Valley,2010
4,2010-05-01,58.154,San Simon Valley,2010
...,...,...,...,...
955,2019-08-01,6.186,Cuyama,2019
956,2019-09-01,7.972,Cuyama,2019
957,2019-10-01,8.413,Cuyama,2019
958,2019-11-01,10.864,Cuyama,2019


In [3]:
# Sum ET by year and site and include months of data 
df_annual = (
    df_monthly.groupby(["site", "year"])["et"]
    .agg(et="sum", number_of_months="count")
    .reset_index()
)
print(df_annual.head())

                      site  year       et  number_of_months
0  Arizona (Willcox Basin)  2010  815.555                12
1  Arizona (Willcox Basin)  2011  835.003                12
2  Arizona (Willcox Basin)  2012  836.401                12
3  Arizona (Willcox Basin)  2013  914.868                12
4  Arizona (Willcox Basin)  2014  941.966                12


In [4]:
df_annual.to_csv("openet_et_annual.csv", index=False)
df_monthly.to_csv("openet_et_monthly.csv", index=False)

In [5]:
df_monthly.groupby('year').size()

year
2010    96
2011    96
2012    96
2013    96
2014    96
2015    96
2016    96
2017    96
2018    96
2019    96
dtype: int64